<a href="https://colab.research.google.com/github/Imaad-0055/Coding-week_gr7_PREDICTING-HEART-FAILURE-RISK-WITH-EXPLAINABLE-ML-SHAP-/blob/main/AI_Agents_for_Financial_Analysis_Part_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to the AI Agents for Financial Analysis Workshop!

## Part 1: Full Context Approach

<a target="_blank" href="https://colab.research.google.com/github/MichaelKarpe/lehman/blob/dev/AI_Agents_for_Financial_Analysis_Part_1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In this workshop, we will build AI agents to investigate Lehman Brothers' SEC filings for signs of bankruptcy.
We will explore two distinct approaches in two separate notebooks:
1. **Full Context (This Notebook)**: Passing the entire SEC filing into the Large Language Model's (LLM) context window. Ideal for comprehensive analysis, leveraging Gemini's massive 2-million token context window.
2. **Retrieval-Augmented Generation (RAG)**: Chunking the filings, embedding them into a vector database, and retrieving only the most relevant sections. Ideal for cost/token optimization and precise source citation.

**⚠️ IMPORTANT**: Participants should go through this Full Context notebook completely before proceeding to the RAG notebook to understand the baseline comparison.

**SEC Filing Example:**
- **PDF**: [https://www.rns-pdf.londonstockexchange.com/rns/8436Z_1-2008-7-24.pdf](https://www.rns-pdf.londonstockexchange.com/rns/8436Z_1-2008-7-24.pdf)
- **Text**: [https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm](https://www.sec.gov/Archives/edgar/data/806085/000110465908045115/a08-18147_110q.htm)

### 🔑 API Setup
1. **Create API key** at the following link: [https://aistudio.google.com/u/1/api-keys](https://aistudio.google.com/u/1/api-keys)
2. When the notebook is opened in Google Colab, click on **"Secrets"** on the left panel, then **"Gemini API keys"** and **"Import key from Google AI Studio"** to get your Gemini API key. Toggle the **"Notebook access"** button to activate it if not done automatically.

In [3]:
!pip install -q edgartools==5.21.1 pydantic-ai==1.67.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.4/93.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.9/567.9 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4

## Notebook setup and basic `edgartools` methods

First, let's setup the notebook for the workshop and investigate a few basic methods of the `edgartools` library, which will be used to retrieve SEC filings for financial analysis.

In [5]:
import os
from typing import Optional
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dataclasses import dataclass
from datetime import datetime
import edgar
from edgar import Company, Filing
import asyncio
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from rich.text import Text

edgar.set_identity("your_email@example.com")

# Set Google API key
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
if not os.getenv("GOOGLE_API_KEY"):
    print("⚠️ GOOGLE_API_KEY not found in environment variables.")
    print("Set it before running the notebook.")

### Example 1: Search for a Company

We search for a specific company using `edgartools`. We can search by its CIK (Central Index Key), which is a unique public number assigned by the SEC to identify a corporation.

> **❓ Question for you:** Why is the CIK preferred over a company name or ticker symbol when programmatically retrieving historical SEC filings, particularly for a company like Lehman Brothers?
> *(Hint: What happens to a ticker when a company gets delisted?)*


In [11]:
# Search for a company

ticker = "LEHKQ"  # update with the ticker of your preferred company still operating today
company = edgar.Company(ticker)
print(f"Company: {company.data.name}")
print(f"CIK: {company.cik}")
ticker = company.get_ticker()
if ticker:
    print(f"Ticker: {ticker}")
else:
    print("Ticker not available for this company.")

CompanyNotFoundError: Company not found: 'LEHKQ'
  Tip: Search by name with find_company("...") or pass a CIK directly.

### Example 2: Get Recent Filings

Now we retrieve the company's SEC filings. Common filing forms include:
* **10-K**: Comprehensive annual financial report.
* **10-Q**: Quarterly financial report (less detailed than 10-K).
* **8-K**: Current report of material events (e.g., bankruptcy, leadership changes, major operations).

> **❓ Question for you:** What type of information would you look for in an 8-K, a 10-Q or a 10-K if you were an analyst searching for sudden red flags?


In [7]:
# Get recent 10-K filings (annual reports)
filings = company.get_filings(form="10-K")
filings

╭────────────────────────────────────── Filings for Alphabet Inc. [1652044] ──────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    10-K        Annual report for public companies                            2026-02-05   0001652044-26-0000…   │
│    10-K        Annual report for public companies                            2025-02-05   0001652044-25-0000…   │
│    10-K        Annual report for public companies                            2024-01-31   0001652044-24-0000…   │
│    10-K        Annual report for public companies                     

## Building a Financial Analyst AI Agent

This main section demonstrates how to build an AI agent using **Gemini** and the `pydantic-ai` framework. The agent will analyze Lehman Brothers' 10-K, 10-Q, and 8-K filings from the critical months leading up to its bankruptcy, seeking key risk indicators like liquidity crunches, over-leveraging, and toxic asset exposure.

In this **Full Context approach**, we will read the raw text of the filings and pass *all* of it directly into Gemini's context window.

> **❓ Question for you:** What are the main advantages and limits (e.g., regarding context window size, API cost, latency, and "lost in the middle" phenomena) of passing entire financial filings to the LLM at once?


### Step 1: Define the Financial Analyst Agent

First, we need to define our financial analyst agent, including its system instructions, and define the information we want our agent to retrieve or to infer from the SEC filings.

> **❓ Question for you:** What libraries are using the `BaseModel` and the `Agent` classes? Why are we using these libraries instead of directly calling an LLM for analyzing filings?

In [12]:
# Define the data models for analysis
class BankruptcyWarningSign(BaseModel):
    """Evidence of bankruptcy warning signs from filings"""
    warning_type: str = Field("Type of warning (e.g., Leverage Ratio, Mortgage Exposure, Capital Shortage, Loss)")
    filing_date: str = Field("Date of the filing")
    filing_type: str = Field("Type of filing (10-K, 10-Q, 8-K)")
    key_metric: str = Field("The specific financial metric or event")
    value: Optional[str] = Field("The quantified value if available")
    severity: int = Field("Severity score 1-10, where 10 is most severe", ge=1, le=10)
    description: str = Field("Description of the warning sign")
    implications: str = Field("Why this warning sign mattered for bankruptcy prediction")
    exact_passage: str = Field("Exact quote or passage from the filing that justifies this finding")
    section_reference: str = Field("Section of filing where this passage appears (e.g., Item 1A Risk Factors, MD&A)")

class FilingAnalysisResult(BaseModel):
    """Results from analyzing a single filing"""
    filing_date: str = Field("Date of the filing")
    filing_type: str = Field("Type of filing")
    filing_url: Optional[str] = Field("URL or reference to the filing")
    warning_signs: list[BankruptcyWarningSign] = Field("List of warning signs found")
    overall_risk_score: int = Field("Overall risk score 1-10 for this filing")
    summary: str = Field("Summary of financial health in this filing")

@dataclass
class FilingDependencies:
    """Dependencies for filing analysis"""
    filing: Filing
    filing_text: str

In [13]:
# Write your financial analyst instructions

financial_analyst_system_prompt = """You are a financial forensic analyst specializing in detecting financial distress warning signs from SEC filings.

Your task is to analyze Lehman Brothers' SEC filings (10-K, 10-Q, 8-K) and identify key warning signs that indicate serious financial distress or bankruptcy risk.

For EACH warning sign you identify, you MUST:
1. Extract an exact quote or passage from the filing that directly supports it
2. Identify the specific section where it appears (e.g., Item 1A, MD&A, Balance Sheet note)
3. Explain what makes this passage a red flag
4. Connect it to potential risk based ONLY on the filing content

Key warning signs to look for:
1. LEVERAGE RATIOS: Look for very high debt-to-capital ratios that suggest fragile balance sheets
2. MORTGAGE EXPOSURE: Track mortgage lending volumes and exposure to mortgage securities
3. CAPITAL SHORTAGES: Look for emergency capital raises or declining capital levels
4. OPERATIONAL LOSSES: Identify significant quarterly or annual losses
5. LIQUIDITY CONCERNS: Look for issues with asset liquidity or funding availability
6. RISK DISCLOSURES: Analyze Item 1A Risk Factors for red flags
7. SEGMENT PERFORMANCE: Track deteriorating performance in key business segments

When creating each BankruptcyWarningSign:
- exact_passage: Quote directly from the filing (50-200 characters, actual text)
- section_reference: Where in the filing this appears (e.g., "Item 1A Risk Factors, page 12")
- description: Why this specific quote/data is concerning
- implications: How this warning sign might indicate serious financial distress

Be specific with numbers and dates. Focus only on evidence you find in the actual filing text provided.
"""

In [14]:
# Choose your free Gemini model. Comment/uncomment the lines below to change the model in case of quota exceeded

# model = 'google-gla:gemini-2.5-flash',
model = 'google-gla:gemini-3-flash-preview'

In [15]:
# Create the financial analyst agent
financial_analyst = Agent(
    model=model,
    deps_type=FilingDependencies,
    output_type=FilingAnalysisResult,
    system_prompt=financial_analyst_system_prompt)

@financial_analyst.tool
async def extract_filing_details(
    ctx: RunContext[FilingDependencies]
) -> str:
    """Extract text content from the filing"""
    return ctx.deps.filing_text

print("✓ Agent ready to analyze SEC filings")

✓ Agent ready to analyze SEC filings


### Step 2: Fetch Lehman Brothers Filings

We will target the specific time frame surrounding the financial crisis for Lehman Brothers.

> **❓ Question for you:** Search for Lehman Brothers' CIK and bankruptcy year and update the code below accordingly. How would you check with the below code that you have found the correct bankruptcy year?

In [24]:
# Use the known CIK for Lehman Brothers Holdings Inc
LEHMAN_CIK = 806085
bankruptcy_year = 2008

# Create a Company object directly with the CIK
lehman = Company(LEHMAN_CIK)

print(f"Company: {lehman.data.name}")
print(f"CIK: {lehman.cik}")
print()

# Get filings from the bankruptcy year
# Focus on 10-K (annual) and 10-Q (quarterly) filings

print("📋 Fetching 10-K and 10-Q filings...")

try:
    # Get 10-K filings
    tenk_filings = lehman.get_filings(form="10-K")
    print(f"Found {len(tenk_filings)} 10-K filings")

    # Get 10-Q filings
    tenq_filings = lehman.get_filings(form="10-Q")
    print(f"Found {len(tenq_filings)} 10-Q filings")

except Exception as e:
    print(f"Error fetching filings: {e}")
    tenk_filings = []
    tenq_filings = []

# Filter for bankruptcy year filings only
def is_in_target_period(filing):
    """Check if filing is from the bankruptcy year"""
    try:
        if filing.filing_date:
            if isinstance(filing.filing_date, str):
                year = int(filing.filing_date[:4])
            else:
                year = filing.filing_date.year
            return year == bankruptcy_year
    except:
        pass
    return False

target_filings = {
    "10-K": [f for f in tenk_filings if is_in_target_period(f)],
    "10-Q": [f for f in tenq_filings if is_in_target_period(f)]
}

print("\n📅 Filings to analyze:")
for form, filings in target_filings.items():
    print(f"\n{form} Filings ({len(filings)} found):")
    for f in sorted(filings, key=lambda x: x.filing_date if x.filing_date else "", reverse=True):
        print(f"  • {f.filing_date}: {f.accession_no}")

Company: LEHMAN BROTHERS HOLDINGS INC. PLAN TRUST
CIK: 806085

📋 Fetching 10-K and 10-Q filings...
Found 12 10-K filings
Found 46 10-Q filings

📅 Filings to analyze:

10-K Filings (1 found):
  • 2008-01-29: 0001104659-08-005476

10-Q Filings (2 found):
  • 2008-07-10: 0001104659-08-045115
  • 2008-04-09: 0001104659-08-023292


### Step 3: Define and Run the AI Agent

**📊 Monitor Quotas:**
Monitor Gemini embedding and LLM models quotas when running the cells calling the models at the following links:
- [https://aistudio.google.com/u/1/rate-limit](https://aistudio.google.com/u/1/rate-limit)
- [https://console.cloud.google.com/apis/api/generativelanguage.googleapis.com/quotas](https://console.cloud.google.com/apis/api/generativelanguage.googleapis.com/quotas)
*(On the second link, sort by descending order of "Current usage percentage" to see your current usage)*

Here we define our `pydantic-ai` Agent. We set the model to `gemini-3-flash-preview` and provide strict system instructions. Most importantly, we force the model to output its analysis into a structured Pydantic model (`FilingAnalysisResult`).

> **❓ Question for you:** Look at the prompt instructions and the `FilingAnalysisResult` schema. How does forcing the LLM to output structured JSON data (like Pydantic models) help us programmatically compare different SEC filings over time compared to free-text analysis?



In [25]:
# Step 3: Run the AI Agent on selected filings

async def analyze_filing(filing: Filing, agent, filing_form: str) -> Optional[FilingAnalysisResult]:
    """Analyze a single filing using the AI agent"""
    try:
        print(f"\n🔍 Analyzing {filing_form} filed on {filing.filing_date}...")

        # Extract text from the filing
        filing_text = ""
        try:
            # The proper way to get filing text in edgartools is filing.text()
            if hasattr(filing, 'text') and callable(filing.text):
                filing_text = filing.text()
            elif filing.documents:
                # Fallback to document if filing.text() doesn't work
                main_doc = filing.documents[0]
                if hasattr(main_doc, 'text'):
                    if isinstance(main_doc.text, str):
                        filing_text = main_doc.text
                    elif callable(main_doc.text):
                        filing_text = main_doc.text()
            print("Filling text length: ", len(filing_text))
        except Exception as e:
            print(f"  (Note: Could not extract full text: {str(e)[:50]})")
            filing_text = f"Filing: {filing_form}\nAccession: {filing.accession_no}\nFiled: {filing.filing_date}"

        if not filing_text or len(filing_text) < 50:
            # If we still don't have meaningful text, create a template response
            filing_text = f"""
LEHMAN BROTHERS {filing_form} FILING
Date Filed: {filing.filing_date}
Accession Number: {filing.accession_no}

This is a {filing_form} filing from Lehman Brothers Holdings Inc.

Key sections to analyze:
- Balance Sheet and Asset Quality
- Leverage Ratios and Capital Structure
- Mortgage-Related Exposure
- Risk Factors and Disclosures
- Liquidity and Funding Sources
- Segment Performance

Please analyze this filing for financial distress warning signs based on the information contained within it.
            """

        # Create dependencies
        deps = FilingDependencies(filing=filing, filing_text=filing_text)

        # Run the agent
        prompt = f"""Analyze this {filing_form} filing from Lehman Brothers to identify financial distress warning signs.

This filing was filed on {filing.filing_date}.

The filing text provided is:
{filing_text}

Based ONLY on this filing content, identify warning signs that suggest financial distress or potential bankruptcy risk:
- High leverage or debt-to-capital ratios
- Mortgage lending exposure and securities holdings
- Capital levels and emergency financing
- Quarterly or annual financial losses
- Liquidity constraints or funding concerns
- Risk factor disclosures
- Deteriorating business segment performance

For each warning sign, provide:
1. Warning type
2. Filing date
3. Filing type (10-K, 10-Q)
4. Key metric or event
5. The value if available
6. Severity (1-10) based on how concerning the finding is
7. Description of the concern
8. Implications based on what the filing reveals

Only identify warning signs that are directly supported by text or data in the filing itself."""

        result = await agent.run(prompt, deps=deps)
        return result.output
    except Exception as e:
        print(f"Error analyzing filing: {e}")
        import traceback
        traceback.print_exc()
        return None

async def run_bankruptcy_investigation():
    """Run the full investigation"""
    console = Console()

    # Header
    header = Text("LEHMAN BROTHERS BANKRUPTCY INVESTIGATION", style="bold white on red")
    console.print(Panel(header, box=box.DOUBLE))
    console.print(Text("\nAnalyzing SEC filings to spot bankruptcy warning signs", style="dim"))
    console.print()

    # Select key filings to analyze (all 10-K and 10-Q)
    filings_to_analyze = []

    # Add all 10-K filings sorted by date
    if target_filings["10-K"]:
        for f in sorted(target_filings["10-K"], key=lambda x: x.filing_date if x.filing_date else "", reverse=True):
            filings_to_analyze.append((f"10-K ({f.filing_date.year if hasattr(f.filing_date, 'year') else f.filing_date[:4]})", f))

    # Add all 10-Q filings sorted by date
    if target_filings["10-Q"]:
        for f in sorted(target_filings["10-Q"], key=lambda x: x.filing_date if x.filing_date else ""):
            filings_to_analyze.append((f"10-Q ({f.filing_date})", f))

    # Run analysis on selected filings
    all_results = []
    for form_label, filing in filings_to_analyze:  # Analyze all filings
        result = await analyze_filing(filing, financial_analyst, form_label)
        if result:
            all_results.append(result)
        # rate limiting: one filing per minute, sleep for a bit more than a minute
        import asyncio
        await asyncio.sleep(65)

    return all_results

# Run the investigation
print("\n⏳ Running AI Agent analysis... (this may take a moment)")
print("=" * 60)

try:
    results = await run_bankruptcy_investigation()
    print(f"\n✓ Analysis complete. Found {len(results)} filing analyses.")
except Exception as e:
    print(f"Error running investigation: {e}")
    import traceback
    traceback.print_exc()
    results = []


⏳ Running AI Agent analysis... (this may take a moment)


╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ LEHMAN BROTHERS BANKRUPTCY INVESTIGATION                                                                        ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

Analyzing SEC filings to spot bankruptcy warning signs


🔍 Analyzing 10-K (2008) filed on 2008-01-29...


Filling text length:  762688

🔍 Analyzing 10-Q (2008-04-09) filed on 2008-04-09...


Filling text length:  441885

🔍 Analyzing 10-Q (2008-07-10) filed on 2008-07-10...


Filling text length:  535624

✓ Analysis complete. Found 3 filing analyses.


### Step 4: Display the Investigation Report

Finally, we will compile the structured insights from the AI agent and format them into a readable report using the `rich` library. This allows us to clearly track warning signs and their severity chronologically across the filings.

> **🔍 Verification Task:**
> Please look for the **"EXACT PASSAGE FROM FILING"** displayed as the output of this cell in the PDF or text formats of the filings (links provided at the beginning of the notebook). Make sure that these exact passages are taken from the original documents and are not hallucinated!


In [27]:
# Step 4: Display and Analyze Results

def display_bankruptcy_investigation_report(results: list[FilingAnalysisResult]):
    """Display investigation results in a formatted report with exact passages"""
    console = Console()

    if not results:
        console.print("[yellow]No results to display[/yellow]")
        return

    # Collect all warning signs
    all_warnings = []
    total_risk_score = 0

    for result in results:
        print(f"\n{'='*80}")
        print(f"FILING ANALYSIS: {result.filing_type} - {result.filing_date}")
        print(f"Overall Risk Score: {result.overall_risk_score}/10")
        print(f"{'='*80}")
        print(f"\nSummary: {result.summary}\n")

        # Safely convert overall_risk_score to an integer
        current_risk_score = 0
        try:
            current_risk_score = int(result.overall_risk_score)
        except (ValueError, TypeError):
            console.print(f"[yellow]Warning: Could not parse overall_risk_score for filing {result.filing_date}. Using 0 instead.[/yellow]")

        total_risk_score += current_risk_score

        if result.warning_signs:
            print(f"⚠️  Warning Signs Found ({len(result.warning_signs)}):\n")

            # Sort by severity
            warnings = sorted(result.warning_signs, key=lambda x: x.severity, reverse=True)

            for idx, warning in enumerate(warnings, 1):
                all_warnings.append(warning)

                severity_indicator = "🔴" if warning.severity >= 8 else "🟠" if warning.severity >= 5 else "🟡"

                print(f"{idx}. {severity_indicator} {warning.warning_type.upper()}")
                print(f"   Severity: {warning.severity}/10")
                print(f"   Metric: {warning.key_metric}")
                if warning.value:
                    print(f"   Value: {warning.value}")
                print(f"   Description: {warning.description}")
                print(f"\n   📄 EXACT PASSAGE FROM FILING:")
                print(f"   \"{warning.exact_passage}\"")
                print(f"   📍 Source: {warning.section_reference}")
                print(f"\n   Why it mattered: {warning.implications}\n")
        else:
            print("No major warning signs detected in this filing.\n")

    # Summary table
    if all_warnings:
        print(f"\n{'='*70}")
        print("BANKRUPTCY WARNING SIGNS SUMMARY")
        print(f"{'='*70}\n")

        # Group by warning type
        warning_types = {}
        for warning in all_warnings:
            if warning.warning_type not in warning_types:
                warning_types[warning.warning_type] = []
            warning_types[warning.warning_type].append(warning)

        # Create summary
        table = Table(title="Warning Signs by Type", box=box.ROUNDED)
        table.add_column("Warning Type", style="cyan")
        table.add_column("Count", style="magenta")
        table.add_column("Max Severity", style="red")
        table.add_column("Key Insight", style="white")

        for warn_type, warnings in sorted(warning_types.items()):
            max_severity = max(w.severity for w in warnings)
            key_insight = warnings[0].implications[:50] + "..." if warnings else ""
            table.add_row(warn_type, str(len(warnings)), str(max_severity), key_insight)

        console.print(table)

    # Overall assessment
    print(f"\n{'='*70}")
    print("OVERALL ASSESSMENT")
    print(f"{'='*70}\n")

    avg_risk = total_risk_score / len(results) if results else 0

    print(f"Average Risk Score: {avg_risk:.1f}/10")
    print(f"Total Warning Signs Identified: {len(all_warnings)}")

    if avg_risk >= 8:
        assessment = "🔴 CRITICAL: Clear bankruptcy risk signals"
    elif avg_risk >= 6:
        assessment = "🟠 HIGH: Significant financial distress indicators"
    elif avg_risk >= 4:
        assessment = "🟡 MODERATE: Notable financial concerns"
    else:
        assessment = "🟢 LOW: Relatively stable financial indicators"

    print(f"\nRisk Assessment: {assessment}\n")

# Display results if available
if results:
    display_bankruptcy_investigation_report(results)
else:
    print("\n⚠️ No analysis results available. Ensure API key is set and filings exist.")


FILING ANALYSIS: 10-K - 2008-01-29
Overall Risk Score: 8/10

Summary: The 2007 10-K reveals a firm under significant stress, despite reporting record net income. The primary warning signs include a dangerous spike in leverage to 30.7x, a 29% collapse in Fixed Income revenues, and a massive reclassification of $11.4 billion in mortgage assets to 'Level III' (mark-to-model) status due to a total lack of market liquidity. Furthermore, the firm faced $21.5 billion in unsecured debt maturities over the following 12 months in an environment management described as a 'significant contraction in available liquidity.'

⚠️  Warning Signs Found (5):

1. 🔴 LEVERAGE RATIO
   Severity: 9/10
   Metric: Leverage Ratio
   Value: 30.7x
   Description: The Firm's total leverage ratio surged from 26.2x to 30.7x in a single year, indicating a dangerously thin equity cushion to absorb asset devaluations.

   📄 EXACT PASSAGE FROM FILING:
   "Leverage ratio is defined as total assets divided by total stockho

Warning: Could not parse overall_risk_score for filing Date of the filing. Using 0 instead.

⚠️  Warning Signs Found (6):

1. 🔴 LEVERAGE RATIO
   Severity: 9/10
   Metric: Leverage Ratio
   Value: 31.7x
   Description: Lehman's total leverage ratio increased to 31.7x, indicating that the firm was operating with a very thin equity cushion relative to its total asset base.

   📄 EXACT PASSAGE FROM FILING:
   "Total assets $786,035 [million] ... Total stockholders’ equity $24,832 [million] ... Leverage ratio 31.7"
   📍 Source: MD&A, Tangible Equity Capital and Capital Ratios, page 72

   Why it mattered: Extremely high leverage means even small declines in asset values can wipe out the firm's equity, making the balance sheet fragile during market volatility.

2. 🔴 SEGMENT PERFORMANCE
   Severity: 9/10
   Metric: Fixed Income Net Revenue Growth
   Value: -88%
   Description: The core Fixed Income business, which historically drove significant revenue, collapsed by 88% due to credit market dislocations and mortgage-related losses.

   📄 EXACT PASSAGE FROM FILING:
   "Capital Market

                                              Warning Signs by Type                                              
╭────────────────────────────────┬───────┬──────────────┬───────────────────────────────────────────────────────╮
│ Warning Type                   │ Count │ Max Severity │ Key Insight                                           │
├────────────────────────────────┼───────┼──────────────┼───────────────────────────────────────────────────────┤
│ Asset Liquidity Concerns       │ 1     │ 9            │ High concentrations of Level 3 assets pose a 'valu... │
│ Capital Shortage               │ 2     │ 8            │ Large, dilutive, and rapid capital raises suggest ... │
│ Hedging Failure                │ 1     │ 9            │ If a firm cannot effectively hedge its illiquid mo... │
│ Leverage Ratio                 │ 2     │ 9            │ At 30.7x leverage, a mere 3.3% decline in the valu... │
│ Liquidity Concerns             │ 2     │ 9            │ Large holdings of physical real estate and related... │
│ Liquidity Risk                 │ 1     │ 8            │ Lehman was highly dependent on the ability to 'rol... │
│ Liquidity/Valuation Risk       │ 1     │ 8            │ The shift to Level III valuation indicates that th... │
│ Mortgage Exposure              │ 2     │ 8            │ The firm remained heavily 'long' on the exact asse... │
│ Operational Loss               │ 1     │ 10           │ A loss of this magnitude suggests that the firm's ... │
│ Operational Loss/Deterioration │ 1     │ 7            │ This represents a fundamental deterioration in Leh... │
│ Operational Losses             │ 1     │ 8            │ Ongoing negative valuation adjustments reduce net ... │
│ Risk Management Failure        │ 1     │ 8            │ Frequent VaR 'breaks' indicate that market volatil... │
│ Segment Performance            │ 1     │ 9            │ A collapse in the primary revenue-generating segme... │
╰────────────────────────────────┴───────┴──────────────┴───────────────────────────────────────────────────────╯


OVERALL ASSESSMENT

Average Risk Score: 5.7/10
Total Warning Signs Identified: 17

Risk Assessment: 🟡 MODERATE: Notable financial concerns

